<a href="https://colab.research.google.com/github/teju917/GENAITUTORIALS050/blob/main/legal_chatbot_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚖️ Building a Basic Chatbot using an LLM – Legal Information Assistant

- **LLM:** `Qwen/Qwen2.5-1.5B-Instruct` (open-source model, runs inside Colab)
- **No API key needed** – the model is downloaded directly from Hugging Face
- **Interface:** simple text chat + Gradio chat window

> ⚠️ This chatbot gives **general legal information only**, not professional legal advice.

**Tip:** Use *Runtime → Change runtime type → T4 GPU* for faster replies (CPU also works, just slower).

In [1]:
!pip install -q transformers accelerate gradio

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype)
model.to(device)
model.eval()

print("Model loaded successfully on:", device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully on: cuda


In [3]:
# System prompt that makes the LLM behave like a legal information assistant

SYSTEM_PROMPT = """You are LegalBuddy, a friendly legal information assistant.

Rules:
1. Answer only questions related to law and legal topics
   (for example: contracts, tenancy and rent, consumer rights, cyber law,
   family law, employment law, traffic rules, police and arrest rights,
   property, intellectual property, RTI and fundamental rights).
2. If the user does not mention a country, assume India.
3. Explain in simple, easy words and use short points or steps.
4. Give general legal information only, not professional legal advice.
5. If a question is not about law, politely say you can only help with legal topics.
6. For serious matters, advise the user to consult a qualified lawyer.
7. Never help with committing crimes or escaping the law illegally.
"""

DISCLAIMER = "\n\n⚠️ *This is general legal information, not legal advice. Please consult a qualified lawyer for your specific situation.*"

print("System prompt ready!")

System prompt ready!


In [4]:
def generate_reply(messages, max_new_tokens=350):
    """messages = list of {"role": "user"/"assistant", "content": "..."}"""

    # keep only the last 6 messages so the prompt stays short
    messages = messages[-6:]

    chat = [{"role": "system", "content": SYSTEM_PROMPT}] + messages

    prompt_text = tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.5,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    # remove the prompt part and decode only the new answer
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return answer + DISCLAIMER

In [5]:
# Quick test
test_messages = [{"role": "user", "content": "What are the basic rights of a tenant?"}]

print(generate_reply(test_messages))

The basic rights of a tenant include:

1. **Privacy**: The right to privacy in one's own rented space.
2. **Security Deposit Return**: The right to receive back the security deposit after moving out, typically minus any deductions for damages.
3. **Housing Conditions**: The right to live in habitable conditions that meet local building codes.
4. **Right to Repair**: The right to request repairs when they become necessary.
5. **Freedom from Discrimination**: The right to be treated fairly without discrimination based on race, gender, religion, etc.
6. **Termination Right**: The right to terminate the lease agreement if certain conditions are met, such as breach of contract by the landlord.
7. **Notice Period**: The right to give notice before ending the lease agreement.
8. **Access Rights**: The right to access common areas within the rental unit.
9. **Respect for Property**: The right to respect for personal belongings within the rented space.
10. **Legal Protection**: The right to see

## Simple Chat in the Notebook
Type your legal question and press Enter. Type `exit` to stop.

In [6]:
conversation = []

print("⚖️ LegalBuddy: Hello! Ask me any legal question. (type 'exit' to stop)\n")

while True:
    user_message = input("You: ")

    if user_message.strip().lower() in ["exit", "quit", "bye"]:
        print("⚖️ LegalBuddy: Goodbye! Stay informed and stay safe.")
        break

    if not user_message.strip():
        continue

    conversation.append({"role": "user", "content": user_message})

    reply = generate_reply(conversation)

    conversation.append({"role": "assistant", "content": reply})

    print("\n⚖️ LegalBuddy:", reply, "\n")

⚖️ LegalBuddy: Hello! Ask me any legal question. (type 'exit' to stop)

You: what are the funadmanetal rifghts

⚖️ LegalBuddy: The fundamental rights of an Indian citizen include:

1. Right to Equality - This means everyone should be treated equally under the law without any discrimination based on race, religion, caste, sex, etc.

2. Right to Freedom - It includes freedom of speech, expression, assembly, movement, and residence within the territory of India.

3. Right against Exploitation - This right protects people from being exploited for labor or services by others.

4. Right to Constitutional Remedies - Citizens have the right to seek redressal through courts if they feel their constitutional rights have been violated.

5. Right to Liberty - This ensures that no person shall be arrested or detained except as permitted by law.

6. Right against Cognizable Criminal Offenses - People cannot be arrested unless there's evidence proving guilt beyond reasonable doubt.

7. Protection of 

## Chat Window using Gradio

In [ ]:
import gradio as gr

def to_text(content):
    # newer Gradio versions may send content as a list of blocks
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict):
                parts.append(str(block.get("text", "")))
            else:
                parts.append(str(block))
        return " ".join(parts)
    return str(content)


def gradio_chat(message, history):

    messages = []

    # works with both Gradio history formats
    for item in history:
        if isinstance(item, dict):
            messages.append({"role": item["role"], "content": to_text(item["content"])})
        else:
            user_text, bot_text = item
            if user_text:
                messages.append({"role": "user", "content": to_text(user_text)})
            if bot_text:
                messages.append({"role": "assistant", "content": to_text(bot_text)})

    messages.append({"role": "user", "content": to_text(message)})

    return generate_reply(messages)


demo = gr.ChatInterface(
    fn=gradio_chat,
    title="⚖️ LegalBuddy – Legal Information Chatbot",
    description="Ask general legal questions (tenancy, consumer rights, cyber law, contracts, etc.). General information only – not legal advice."
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9492c3b69841508232.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
